# Chapter 15: MLOps & Model Deployment
## Notebook 01 — Packaging & Serving

This notebook walks the **first half of the production lifecycle**: taking a trained model and turning it into a runnable, observable service.

### What you'll learn

| Topic | Section |
|-------|--------|
| The MLOps lifecycle (train -> package -> serve -> monitor -> improve) | §1 |
| Serializing a sklearn pipeline with `joblib` | §2 |
| Typed request / response schemas with Pydantic v2 | §3 |
| A FastAPI service with `/predict`, `/health`, `/version` | §4 |
| Batching, async, and latency budgeting | §5 |
| Containerization concepts and a Dockerfile from scratch | §6 |
| Health checks vs. readiness probes | §7 |

**Estimated time:** 2 hours

---
*Generated by Berta AI*

---
## 1. The MLOps Lifecycle

A production model lives in a loop, not a notebook:

```
train  ->  package  ->  deploy  ->  monitor  ->  improve  ->  (back to train)
```

Each arrow is an artifact contract. *Train* produces a serialized model + metrics. *Package* wraps it with a schema and dependencies. *Deploy* exposes it behind an API with a version label. *Monitor* watches inputs, outputs, latency, and errors. *Improve* triggers re-training when something drifts or breaks.

This notebook covers the first three steps. Notebook 02 covers pipelines and CI/CD; Notebook 03 covers monitoring and operating models at scale.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

import config
print('chapter root:', config.chapter_root())
print('default model file:', config.DEFAULT_MODEL_FILE)
print('p95 latency budget (ms):', config.P95_LATENCY_MS)

---
## 2. Train and Serialize a Pipeline

We train a tiny scikit-learn `Pipeline` (scaler + logistic regression) on synthetic data and serialize it with **joblib**. Joblib is preferred over `pickle` for sklearn because it stores large NumPy arrays efficiently.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(42)
n = 400
X = rng.normal(size=(n, 2))
# Decision boundary: x0 + 0.5*x1 > 0
y = ((X[:, 0] + 0.5 * X[:, 1]) > 0).astype(int)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=200, random_state=42)),
])
pipe.fit(X, y)
print('train accuracy:', round(pipe.score(X, y), 3))

In [ ]:
# Serialize and reload — the round-trip must be exact.
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)
artifact = MODEL_DIR / 'logreg_v0.1.0.joblib'

joblib.dump(pipe, artifact)
print('artifact size (bytes):', artifact.stat().st_size)

reloaded = joblib.load(artifact)
print('reloaded predict[:5]:', reloaded.predict(X[:5]))
print('original predict[:5]:', pipe.predict(X[:5]))
assert (reloaded.predict(X) == pipe.predict(X)).all(), 'round-trip mismatch'

**Production hygiene:**
- Pin `joblib` and `scikit-learn` in `requirements.txt` — pickled pipelines are sensitive to library versions.
- Store the artifact alongside its **metadata** (training date, metrics, data hash, code commit). We'll wire this up in §6 of Notebook 02 (the registry).
- Never deserialize artifacts from untrusted sources: `joblib.load` executes pickled code.

---
## 3. Pydantic Schemas: Typed I/O

Every production endpoint should have an **explicit schema**. Pydantic v2 gives us free validation, automatic OpenAPI docs (via FastAPI), and clear errors at the boundary.

In [ ]:
from pydantic import BaseModel, Field, ConfigDict, ValidationError

class PredictRequest(BaseModel):
    model_config = ConfigDict(extra='forbid')
    feature_a: float = Field(..., description='Numeric feature A.')
    feature_b: float = Field(..., description='Numeric feature B.')

# Valid request
ok = PredictRequest(feature_a=0.1, feature_b=-0.3)
print('valid:', ok.model_dump())

# Invalid: extra field
try:
    PredictRequest(feature_a=0.1, feature_b=0.0, sneaky='oops')
except ValidationError as e:
    print('extra field rejected:', e.errors()[0]['type'])

# Invalid: wrong type
try:
    PredictRequest(feature_a='not-a-float', feature_b=0.0)
except ValidationError as e:
    print('type rejected:', e.errors()[0]['type'])

**Why typed schemas matter in production:**

1. **Fail fast at the boundary** — bad inputs never reach the model.
2. **OpenAPI docs come free** — FastAPI auto-generates interactive docs at `/docs`.
3. **Clients get a contract** — codegen tools can produce typed SDKs from the schema.
4. **Drift detection becomes easier** — schemas double as the source of truth for monitored fields.

---
## 4. The FastAPI Service

We use `scripts/deployment.py` which exposes `ModelService` and `build_app()`. Endpoints:

- `GET /health` — liveness (does the process answer?)
- `GET /version` — what is currently deployed?
- `POST /predict` — single-record prediction
- `POST /predict/batch` — vectorized batch prediction

We exercise the app **in-process** using `fastapi.testclient.TestClient` — no port binding, no async loop to babysit, and the same code runs identically under `uvicorn` in production.

In [ ]:
from deployment import ModelService, build_app
from fastapi.testclient import TestClient

service = ModelService(
    model=reloaded,
    name='demo-classifier',
    version='0.1.0',
    stage='Staging',
    framework='sklearn',
)
app = build_app(service)
client = TestClient(app)

print('GET /health  ->', client.get('/health').json())
print('GET /version ->', client.get('/version').json())

In [ ]:
# Single prediction
resp = client.post('/predict', json={'feature_a': 0.4, 'feature_b': -0.1})
print('status:', resp.status_code)
print('body:', resp.json())

# Batch prediction
batch = {'records': [
    {'feature_a': 0.1, 'feature_b': 0.2},
    {'feature_a': -1.0, 'feature_b': 0.3},
    {'feature_a': 0.7, 'feature_b': -0.5},
]}
resp_b = client.post('/predict/batch', json=batch)
print('batch:', resp_b.json())

In [ ]:
# Validation errors return HTTP 422 — try it.
bad = client.post('/predict', json={'feature_a': 'not-a-number', 'feature_b': 0.0})
print('status:', bad.status_code)
print('detail[0]:', bad.json()['detail'][0]['type'], '|', bad.json()['detail'][0]['msg'])

---
## 5. Batching, Async, and Latency Budgeting

A model can usually amortize the cost of one prediction over many. Two reasons your endpoint should support batches:

1. **Throughput** — vectorized NumPy is dramatically faster than a Python loop of single calls.
2. **Inference cost** — for GPU-served models a batch of 32 may be ~30x faster per row than 32 sequential calls.

**Latency budget** is the time the *user* will wait. Decompose it:

```
total = network_in + queue + preprocess + inference + postprocess + network_out
```

Each step should have a budget. p95 < 200 ms is a common target for interactive models.

In [ ]:
# Compare single calls vs. batch for 100 predictions.
N = 100
records = [{'feature_a': float(rng.normal()), 'feature_b': float(rng.normal())} for _ in range(N)]

t0 = time.perf_counter()
for r in records:
    client.post('/predict', json=r)
t_single = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
client.post('/predict/batch', json={'records': records})
t_batch = (time.perf_counter() - t0) * 1000
print(f'100 single calls : {t_single:7.1f} ms total | per-call: {t_single/N:5.2f} ms')
print(f'1 batch of 100   : {t_batch:7.1f} ms total | per-row : {t_batch/N:5.2f} ms')
print(f'speedup          : {t_single / max(t_batch, 1e-6):5.1f}x')

**FastAPI is async-friendly.** Define endpoints as `async def` when they do I/O (DB lookups, RPC calls). For CPU-bound inference, prefer **process workers** (`uvicorn --workers N`) so one slow request doesn't block the event loop. Mark long-running calls with `run_in_threadpool`.

---
## 6. Containerization: Dockerfile from Scratch

A production service ships as a **container image**. Below is a minimal multi-stage Dockerfile for the FastAPI service. We don't run `docker build` here — we just author and inspect the file.

In [ ]:
DOCKERFILE = '''
# syntax=docker/dockerfile:1.6

# ---- builder ----
FROM python:3.11-slim AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# ---- runtime ----
FROM python:3.11-slim
WORKDIR /app
# Copy installed deps from builder layer
COPY --from=builder /root/.local /root/.local
ENV PATH=/root/.local/bin:$PATH

# Copy app code and the model artifact
COPY scripts ./scripts
COPY models ./models

# Run as a non-root user
RUN useradd --uid 10001 --no-create-home appuser
USER appuser

EXPOSE 8000
HEALTHCHECK --interval=15s --timeout=3s --retries=3 \\
    CMD python -c "import httpx, sys; sys.exit(0 if httpx.get('http://localhost:8000/health').json().get('ready') else 1)"

CMD ["uvicorn", "scripts.deployment:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
'''.strip()

print(DOCKERFILE)
print('\n--- layer count ---')
print('directives:',
      sum(1 for line in DOCKERFILE.splitlines() if line.split() and line.split()[0] in {'FROM','COPY','RUN','CMD','ENV','EXPOSE','USER','WORKDIR','HEALTHCHECK'}))

**Layer hygiene checklist:**

- **Order from least -> most volatile.** `requirements.txt` changes rarely; code changes often. Copy and install deps *before* copying source so Docker can cache the dependency layer.
- **Multi-stage builds** keep the final image lean (no compilers, no `pip` cache).
- **Pin the base image** to a digest (`python:3.11-slim@sha256:...`) for true reproducibility.
- **Run as non-root** — minimizes blast radius if the process is compromised.
- **`HEALTHCHECK`** lets the orchestrator (Docker, Kubernetes) decide when to restart or stop routing traffic.

---
## 7. Health vs. Readiness

Two distinct probes, often confused:

| Probe | Question | Failure action |
|-------|---------|----------------|
| **Liveness** (`/health`) | Is the process alive and responsive? | Orchestrator restarts the pod. |
| **Readiness** (`/ready`)  | Is it ready to serve real traffic? | Orchestrator stops routing traffic, but does not restart. |

A model service often passes liveness *long before* it's ready: the process is up, but the model artifact is still loading from object storage. Treating "process up" as "ready" leads to 5xx storms during rollouts.

Our `/health` endpoint already reports a `ready` flag; in a real deployment you'd add a separate `/ready` route that returns 503 until `model.load()` finishes.

In [ ]:
# Quick check: ModelService can be put in a 'not ready' state and the
# endpoint refuses traffic with a clear 503.
service._ready = False
not_ready = client.post('/predict', json={'feature_a': 0.1, 'feature_b': 0.2})
print('status when not ready:', not_ready.status_code, '|', not_ready.json())
service._ready = True  # restore for downstream cells

---
## 8. Key Takeaways

- **Joblib** for sklearn artifacts; pin versions; never deserialize untrusted pickles.
- **Pydantic v2** schemas catch bad inputs at the boundary and give you free OpenAPI docs.
- **FastAPI + TestClient** lets you test the whole service in-process — same code, same paths.
- **Batch endpoints** unlock throughput; budget total latency, not just inference time.
- **Dockerfiles**: order layers least -> most volatile, multi-stage, non-root, with a HEALTHCHECK.
- **Liveness != readiness**: model loading is a real readiness gate.

Next: **Notebook 02** — sklearn pipelines, reproducibility, experiment tracking, a model registry, and CI/CD.

---
*Generated by Berta AI*